# Monte Carlo Estimators

TARDIS Monte Carlo simulations produce three types of estimators:
- **`EstimatorsBulk`**: Cell-level radiation field (mean intensity, mean frequency)
- **`EstimatorsLine`**: Line interactions (J_blue, energy deposition)
- **`EstimatorsContinuum`**: Continuum processes (photoionization, heating)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tardis import run_tardis
from tardis.transport.montecarlo.montecarlo_main_loop import montecarlo_main_loop

%matplotlib inline

## Run Full Simulation

In [ ]:
sim = run_tardis("tardis_example")

## Run Monte Carlo Loop Directly

Now run just the Monte Carlo transport to generate fresh estimators.

In [ ]:
# Extract necessary components from simulation
packet_source = sim.transport.packet_source
geometry_state = sim.simulation_state.geometry.to_numba()
time_explosion = sim.simulation_state.time_explosion
opacity_state = sim.plasma.opacity_state
montecarlo_configuration = sim.transport.montecarlo_configuration
number_of_vpackets = sim.transport.number_of_vpackets

# Run Monte Carlo loop
(
    v_packets_energy_hist,
    vpacket_tracker,
    estimators_bulk,
    estimators_line,
    estimators_continuum,
) = montecarlo_main_loop(
    packet_source,
    geometry_state,
    time_explosion,
    opacity_state,
    montecarlo_configuration,
    number_of_vpackets,
)

print(f"Generated estimators from {montecarlo_configuration.number_of_packets} packets")

## Examine Estimators

In [ ]:
print(f"Bulk: mean_intensity_total shape = {estimators_bulk.mean_intensity_total.shape}")
print(f"Line: mean_intensity_blue shape = {estimators_line.mean_intensity_blue.shape}")
print(f"Continuum: photo_ion_estimator shape = {estimators_continuum.photo_ion_estimator.shape}")

## Visualize Results

In [ ]:
N_CELLS = len(estimators_bulk.mean_intensity_total)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(estimators_bulk.mean_intensity_total, marker='o')
ax1.set_xlabel('Cell Index')
ax1.set_ylabel('Mean Intensity')
ax1.set_title('J vs Cell')
ax1.grid(alpha=0.3)

ax2.plot(estimators_bulk.mean_frequency, marker='s', color='orange')
ax2.set_xlabel('Cell Index')
ax2.set_ylabel('Mean Frequency (Hz)')
ax2.set_title('$\\bar{\\nu}$ vs Cell')
ax2.grid(alpha=0.3)

plt.tight_layout()

In [ ]:
# Plot J_blue for representative lines
N_LINES = estimators_line.mean_intensity_blue.shape[0]
LINE_INDICES = [0, N_LINES // 4, N_LINES // 2, 3 * N_LINES // 4, N_LINES - 1]

fig, ax = plt.subplots(figsize=(10, 5))

for idx in LINE_INDICES:
    ax.plot(estimators_line.mean_intensity_blue[idx, :], marker='o', label=f'Line {idx}')

ax.set_xlabel('Cell Index')
ax.set_ylabel('$J_{\\mathrm{blue}}$')
ax.set_title('Line Intensity Across Cells')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()